## Notebook 2/3 — Reddit: Train (5-fold CV) → Evaluate → Visualize → Comparison Table

**Prerequisite**: run `01_reddit_preprocess_embed.ipynb` first.

This notebook reads artifacts from `outputs/` and produces:
- `results_comparison.csv`
- confusion matrices + plots (PNG)
- optional `reddit_cv_metrics_by_fold.csv` / `reddit_cv_summary.csv`

In [4]:
import os
from pathlib import Path

# Check current working directory
print("Current directory:", Path.cwd())

# Search for the embeddings file anywhere in the project
for p in Path(".").rglob("reddit_embeddings.npy"):
    print("Found:", p.resolve())

# List what's in outputs/ if it exists
outputs = Path("outputs")
if outputs.exists():
    print("\nContents of outputs/:")
    for f in outputs.iterdir():
        print(" ", f)
else:
    print("\nNo outputs/ folder found in current directory")

Current directory: c:\Users\HP\Desktop\major2
Found: C:\Users\HP\Desktop\major2\outputs\reddit_embeddings.npy

Contents of outputs/:
  outputs\reddit_cleaned_balanced.csv
  outputs\reddit_embeddings.npy
  outputs\reddit_labels.npy


In [5]:
# Colab (optional)
!pip -q install -U pandas numpy scikit-learn sentence-transformers matplotlib seaborn

from pathlib import Path
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from pipeline_utils import (
    LABEL_NAME,
    build_models,
    evaluate_models_cv,
    RANDOM_STATE,
)

sns.set_theme(style="whitegrid")

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

emb_path = OUTPUT_DIR / "reddit_embeddings.npy"
lab_path = OUTPUT_DIR / "reddit_labels.npy"
clean_csv_path = OUTPUT_DIR / "reddit_cleaned_balanced.csv"

if not emb_path.exists() or not lab_path.exists():
    raise FileNotFoundError(
        "Missing Reddit artifacts. Run 01_reddit_preprocess_embed.ipynb first to create outputs/reddit_embeddings.npy and outputs/reddit_labels.npy"
    )

X = np.load(emb_path)
y = np.load(lab_path)

print("Loaded:", emb_path.resolve())
print("Loaded:", lab_path.resolve())
print("X shape:", X.shape, "y shape:", y.shape)

Loaded: C:\Users\HP\Desktop\major2\outputs\reddit_embeddings.npy
Loaded: C:\Users\HP\Desktop\major2\outputs\reddit_labels.npy
X shape: (4000, 384) y shape: (4000,)



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
def save_fig(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()


def plot_confusion_matrix_heatmap(cm: np.ndarray, title: str, out_path: Path, label_names: list[str], normalize: bool = False) -> None:
    cm_plot = cm.astype(float)
    if normalize:
        row_sums = cm_plot.sum(axis=1, keepdims=True)
        cm_plot = np.divide(cm_plot, row_sums, out=np.zeros_like(cm_plot), where=row_sums != 0)

    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    fmt = ".2f" if normalize else ".0f"
    sns.heatmap(cm_plot, annot=True, fmt=fmt, cmap="Blues", xticklabels=label_names, yticklabels=label_names, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    save_fig(out_path)


label_order = [0, 1]
label_names = [LABEL_NAME[i] for i in label_order]

### Step 5 — 5-fold CV (LR / SVM / MLP)

Collects metrics per fold + aggregated confusion matrices.

In [9]:
models = build_models()
metrics_df, cm_sum_by_model, cm_avg_by_model = evaluate_models_cv(X, y, models, n_splits=5)

print("Per-fold metrics (first 10 rows):")
display(metrics_df.head(10))

metrics_df.to_csv(OUTPUT_DIR / "reddit_cv_metrics_by_fold.csv", index=False)

results_table = (
    metrics_df.groupby("model")
    .agg(
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        precision_weighted_mean=("precision_weighted", "mean"),
        precision_weighted_std=("precision_weighted", "std"),
        recall_weighted_mean=("recall_weighted", "mean"),
        recall_weighted_std=("recall_weighted", "std"),
        f1_weighted_mean=("f1_weighted", "mean"),
        f1_weighted_std=("f1_weighted", "std"),
        precision_macro_mean=("precision_macro", "mean"),
        precision_macro_std=("precision_macro", "std"),
        recall_macro_mean=("recall_macro", "mean"),
        recall_macro_std=("recall_macro", "std"),
        f1_macro_mean=("f1_macro", "mean"),
        f1_macro_std=("f1_macro", "std"),
    )
    .reset_index()
    .sort_values("accuracy_mean", ascending=False)
)

results_table.to_csv(OUTPUT_DIR / "reddit_cv_summary.csv", index=False)

print("\nResults summary table:")
display(results_table)

Per-fold metrics (first 10 rows):


,model,fold,accuracy,precision_weighted,recall_weighted,f1_weighted,precision_macro,recall_macro,f1_macro
0,LR,1,0.77000,0.770061,0.77000,0.769987,0.770061,0.77000,0.769987
1,LR,2,0.75000,0.750025,0.75000,0.749994,0.750025,0.75000,0.749994
2,LR,3,0.74375,0.743934,0.74375,0.743702,0.743934,0.74375,0.743702
3,LR,4,0.75875,0.759608,0.75875,0.758550,0.759608,0.75875,0.758550
4,LR,5,0.76375,0.764784,0.76375,0.763519,0.764784,0.76375,0.763519
5,SVM,1,0.80500,0.805000,0.80500,0.805000,0.805000,0.80500,0.805000
6,SVM,2,0.77875,0.778766,0.77875,0.778747,0.778766,0.77875,0.778747
7,SVM,3,0.77000,0.770108,0.77000,0.769977,0.770108,0.77000,0.769977
8,SVM,4,0.78125,0.781646,0.78125,0.781173,0.781646,0.78125,0.781173
9,SVM,5,0.80875,0.808845,0.80875,0.808735,0.808845,0.80875,0.808735



Results summary table:


,model,accuracy_mean,accuracy_std,precision_weighted_mean,precision_weighted_std,recall_weighted_mean,recall_weighted_std,f1_weighted_mean,f1_weighted_std,precision_macro_mean,precision_macro_std,recall_macro_mean,recall_macro_std,f1_macro_mean,f1_macro_std
2,SVM,0.78875,0.017116,0.788873,0.017069,0.78875,0.017116,0.788726,0.017127,0.788873,0.017069,0.78875,0.017116,0.788726,0.017127
1,MLP,0.76400,0.009117,0.764285,0.009236,0.76400,0.009117,0.763938,0.009096,0.764285,0.009236,0.76400,0.009117,0.763938,0.009096
0,LR,0.75725,0.010510,0.757683,0.010665,0.75725,0.010510,0.757150,0.010481,0.757683,0.010665,0.75725,0.010510,0.757150,0.010481


### Step 6 — DAIC-WOZ vs Reddit comparison table

DAIC-WOZ rows are pre-filled from Part-1 (static), and Reddit rows come from the CV results above.

Saved as `outputs/results_comparison.csv`.

In [11]:
reddit_model_results = results_table.set_index("model")[["accuracy_mean", "f1_weighted_mean", "f1_macro_mean"]].copy()

DAIC_STATIC_ROWS = [
    {"Model": "LR", "Dataset": "DAIC-WOZ", "Accuracy": 0.65, "Weighted F1": 0.51, "Macro F1": 0.39},
    {"Model": "SVM/QSVC", "Dataset": "DAIC-WOZ", "Accuracy": 0.47, "Weighted F1": 0.44, "Macro F1": 0.44},
    {"Model": "QSVM", "Dataset": "DAIC-WOZ", "Accuracy": 0.47, "Weighted F1": 0.44, "Macro F1": 0.44},
]

reddit_rows = []
for model_name in ["LR", "SVM", "MLP"]:
    if model_name not in reddit_model_results.index:
        continue
    reddit_rows.append(
        {
            "Model": model_name,
            "Dataset": "Reddit",
            "Accuracy": float(reddit_model_results.loc[model_name, "accuracy_mean"]),
            "Weighted F1": float(reddit_model_results.loc[model_name, "f1_weighted_mean"]),
            "Macro F1": float(reddit_model_results.loc[model_name, "f1_macro_mean"]),
        }
    )

comparison_df = pd.DataFrame(DAIC_STATIC_ROWS + reddit_rows)

display(comparison_df)

comparison_path = OUTPUT_DIR / "results_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)
print("Saved:", comparison_path.resolve())

,Model,Dataset,Accuracy,Weighted F1,Macro F1
0,LR,DAIC-WOZ,0.65000,0.510000,0.390000
1,SVM/QSVC,DAIC-WOZ,0.47000,0.440000,0.440000
2,QSVM,DAIC-WOZ,0.47000,0.440000,0.440000
3,LR,Reddit,0.75725,0.757150,0.757150
4,SVM,Reddit,0.78875,0.788726,0.788726
5,MLP,Reddit,0.76400,0.763938,0.763938


Saved: C:\Users\HP\Desktop\major2\outputs\results_comparison.csv


### Step 7 — Visualizations

Saves:
- class distribution (before/after balancing) *(if cleaned CSV exists)*
- confusion matrix heatmaps (avg + row-normalized)
- weighted F1 comparison bar chart (DAIC-WOZ vs Reddit)
- post length histogram (if cleaned CSV exists)

All plots saved as PNG under `outputs/`.

In [12]:
# Confusion matrices
for model_name, cm_avg in cm_avg_by_model.items():
    plot_confusion_matrix_heatmap(
        cm_avg,
        title=f"{model_name} confusion matrix (avg counts across folds)",
        out_path=OUTPUT_DIR / f"cm_{model_name}_avg_counts.png",
        label_names=label_names,
        normalize=False,
    )
    plot_confusion_matrix_heatmap(
        cm_avg,
        title=f"{model_name} confusion matrix (row-normalized)",
        out_path=OUTPUT_DIR / f"cm_{model_name}_row_normalized.png",
        label_names=label_names,
        normalize=True,
    )


# F1 comparison bar chart
fig, ax = plt.subplots(figsize=(8, 4))
plot_df = comparison_df.copy()
plot_df["Model_plot"] = plot_df["Model"].replace({"SVM/QSVC": "SVM"})

sns.barplot(data=plot_df, x="Model_plot", y="Weighted F1", hue="Dataset", ax=ax)
ax.set_title("Weighted F1: DAIC-WOZ vs Reddit")
ax.set_xlabel("Model")
ax.set_ylabel("Weighted F1")
ax.legend(title="Dataset")

save_fig(OUTPUT_DIR / "weighted_f1_daic_vs_reddit.png")


# Optional: class distribution + length hist (needs cleaned CSV)
if clean_csv_path.exists():
    cleaned = pd.read_csv(clean_csv_path)

    # Post length distribution histogram by risk label
    fig, ax = plt.subplots(figsize=(9, 4.5))
    cleaned["risk_label_name"] = pd.Categorical(
        cleaned["risk_label_name"],
        categories=["Control", "Mental Health Risk", "High Risk"],
        ordered=True,
    )

    sns.histplot(
        data=cleaned,
        x="word_count",
        hue="risk_label_name",
        bins=50,
        element="step",
        stat="density",
        common_norm=False,
        ax=ax,
    )
    ax.set_title("Post length distribution (word count) by risk label")
    ax.set_xlabel("Word count")
    ax.set_ylabel("Density")
    ax.set_xlim(0, np.percentile(cleaned["word_count"], 99))

    save_fig(OUTPUT_DIR / "post_length_by_label.png")

print("Saved plots to:", OUTPUT_DIR.resolve())

Saved plots to: C:\Users\HP\Desktop\major2\outputs
